In [ ]:
def generate_tree_string(dir_path, prefix=""):
    """Standard ASCII characters use karke tree generate karta hai (PDF font safe)"""
    tree_str = ""
    try:
        entries = sorted(os.listdir(dir_path))
        entries = [e for e in entries if e not in IGNORE_FOLDERS]
        
        count = len(entries)
        for i, entry in enumerate(entries):
            path = os.path.join(dir_path, entry)
            is_last = (i == count - 1)
            
            # Standard ASCII characters (PDF font compatible)
            connector = "`-- " if is_last else "|-- "
            
            if os.path.isdir(path):
                tree_str += f"{prefix}{connector}{entry}/\n"
                new_prefix = prefix + ("    " if is_last else "|   ")
                tree_str += generate_tree_string(path, new_prefix)
            else:
                tree_str += f"{prefix}{connector}{entry}\n"
    except Exception as e:
        tree_str += f"{prefix}[Error listing directory: {e}]\n"
        
    return tree_str

In [1]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted, Image, PageBreak
from reportlab.lib.units import inch

# 1. Apne Django Project Ka Path Yahan Dalein
PROJECT_FOLDER = r"C:\path\to\your\django_project"
OUTPUT_PDF = "Django_Project_With_Tree.pdf"

# Unwanted folders jinhe ignore karna hai
IGNORE_FOLDERS = {'node_modules', '__pycache__', '.git', 'venv', '.idea', '.vscode'}

# In Extensions ka DATA read karke PDF me likha jayega
TEXT_EXTENSIONS = (
    '.py', '.js', '.jsx', '.ts', '.tsx', '.vue',
    '.html', '.css', '.scss', '.json', '.yaml', '.yml',
    '.md', '.txt', '.sql', '.xml', '.ini', '.cfg', '.env', '.gitignore'
)

IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg')

def generate_tree_string(dir_path, prefix=""):
    """Standard ASCII characters use karke tree generate karta hai (PDF font safe)"""
    tree_str = ""
    try:
        entries = sorted(os.listdir(dir_path))
        entries = [e for e in entries if e not in IGNORE_FOLDERS]
        
        count = len(entries)
        for i, entry in enumerate(entries):
            path = os.path.join(dir_path, entry)
            is_last = (i == count - 1)
            
            # Standard ASCII characters (PDF font compatible)
            connector = "`-- " if is_last else "|-- "
            
            if os.path.isdir(path):
                tree_str += f"{prefix}{connector}{entry}/\n"
                new_prefix = prefix + ("    " if is_last else "|   ")
                tree_str += generate_tree_string(path, new_prefix)
            else:
                tree_str += f"{prefix}{connector}{entry}\n"
    except Exception as e:
        tree_str += f"{prefix}[Error listing directory: {e}]\n"
        
    return tree_str

def build_pdf():
    doc = SimpleDocTemplate(
        OUTPUT_PDF,
        pagesize=letter,
        rightMargin=36, leftMargin=36,
        topMargin=36, bottomMargin=36
    )
    
    story = []
    styles = getSampleStyleSheet()

    # Front Page Styles
    main_title_style = ParagraphStyle(
        'MainTitle',
        parent=styles['Heading1'],
        fontSize=20,
        leading=24,
        textColor=colors.HexColor('#0F172A'),
        spaceAfter=15,
        alignment=1 # Center align
    )

    tree_style = ParagraphStyle(
        'TreeStyle',
        fontName='Courier',
        fontSize=8.5,
        leading=11,
        textColor=colors.HexColor('#0F172A'),
        backColor=colors.HexColor('#F8FAFC'),
        borderColor=colors.HexColor('#CBD5E1'),
        borderWidth=1,
        borderPadding=10,
        spaceAfter=15
    )

    # Code/Data Styles
    file_title_style = ParagraphStyle(
        'FileHeader',
        parent=styles['Heading2'],
        fontSize=11,
        leading=13,
        textColor=colors.HexColor('#1E293B'),
        backColor=colors.HexColor('#E2E8F0'),
        borderPadding=5,
        spaceAfter=8
    )

    code_style = ParagraphStyle(
        'CodeBody',
        fontName='Courier',
        fontSize=7.5,
        leading=9.5,
        textColor=colors.HexColor('#1E1E1E')
    )

    # -------------------------------------------------------------
    # FRONT PAGE: PROJECT TITLE & TREE STRUCTURE
    # -------------------------------------------------------------
    folder_name = os.path.basename(os.path.normpath(PROJECT_FOLDER))
    story.append(Paragraph(f"<b>PROJECT DIRECTORY TREE</b><br/><font size=12 color='#64748B'>Folder: {folder_name}</font>", main_title_style))
    story.append(Spacer(1, 10))

    print("Generating Tree Structure for Front Page...")
    tree_text = f"📁 {folder_name}/\n" + generate_tree_string(PROJECT_FOLDER)
    
    # Preformatted style Tree Structure ko Front Page par add karega
    story.append(Preformatted(tree_text, tree_style))
    
    # Front Page Complete -> Next Page par jumping
    story.append(PageBreak())

    # -------------------------------------------------------------
    # MAIN CONTENT: FILES & FOLDER DATA WRITING
    # -------------------------------------------------------------
    print("Reading and writing file contents...")

    for root, dirs, files in os.walk(PROJECT_FOLDER):
        # Unwanted folders ko dynamically ignore list me add karna
        dirs[:] = [d for d in dirs if d not in IGNORE_FOLDERS]
        dirs.sort()  # Sequence wise folders
        files.sort() # Sequence wise files

        for file in files:
            file_path = os.path.join(root, file)
            relative_path = os.path.relpath(file_path, PROJECT_FOLDER)
            
            # File Path Header Box
            story.append(Paragraph(f"<b>File Path:</b> {relative_path}", file_title_style))

            # Option A: Text/Code files data write karna
            if file.endswith(TEXT_EXTENSIONS) or file.startswith('.env'):
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
                        content = f.read()
                        
                    if content.strip():
                        story.append(Preformatted(content, code_style))
                    else:
                        story.append(Paragraph("<i>[Empty File]</i>", styles['Italic']))

                except Exception as e:
                    story.append(Paragraph(f"<i>[Error reading file: {e}]</i>", styles['Italic']))

            # Option B: Images write karna
            elif file.endswith(IMAGE_EXTENSIONS):
                try:
                    img = Image(file_path)
                    img.drawWidth = 4 * inch
                    img.drawHeight = 2.5 * inch
                    story.append(img)
                except Exception as e:
                    story.append(Paragraph(f"<i>[Error rendering image: {e}]</i>", styles['Italic']))

            story.append(Spacer(1, 12))

    doc.build(story)
    print(f"\n✅ SUCCESS! Project Tree aur Data '{OUTPUT_PDF}' me successfully generate ho gaya hai.")

if __name__ == "__main__":
    build_pdf()

Generating Tree Structure for Front Page...
Reading and writing file contents...

✅ SUCCESS! Project Tree aur Data 'Django_Project_With_Tree.pdf' me successfully generate ho gaya hai.
